# File search: glob and grep

`FilesystemFileSearchMiddleware` searches real files under a configured root. Use a small fixture workspace to see glob matching and content search. This is a tool boundary, not a general OS sandbox.

[File search](https://docs.langchain.com/oss/python/langchain/middleware/built-in#file-search)

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from lab.sandbox import workspace
from langchain.agents import create_agent
from langchain.agents.middleware import FilesystemFileSearchMiddleware
from model_config import get_model, run_config
from lab.faults import ScriptedModel, tool_request
from langchain_core.messages import AIMessage


In [ ]:
# Execute
root = workspace()
print("Search root:", root)
agent = create_agent(
    model=get_model(),
    middleware=[
        FilesystemFileSearchMiddleware(root_path=str(root), use_ripgrep=False),
        TraceMiddleware("14_file_search", pretty=True),
    ],
)
result = agent.invoke({"messages": [("user", prompt("file_search"))]}, config=run_config())

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(result["messages"], pretty=True)

### Glob and all Grep output modes

Glob selects paths with filename patterns. Grep searches content using regular expressions and an optional filename `include` filter. Its output can contain matching file names, matching lines, or match counts. The root bounds the search; the file-size cap excludes oversized files. Ripgrep and the Python fallback implement the same search interface.

In [ ]:
# Execute
search_modes_agent = create_agent(model=ScriptedModel(responses=[
    tool_request("glob_search", pattern="**/*.txt"),
    tool_request("grep_search", pattern="(?i)remote|leave", include="*.txt", output_mode="files_with_matches"),
    tool_request("grep_search", pattern="(?i)remote|leave", include="*.txt", output_mode="content"),
    tool_request("grep_search", pattern="(?i)remote|leave", include="*.txt", output_mode="count"),
    AIMessage(content="Search modes complete."),
]), middleware=[FilesystemFileSearchMiddleware(root_path=str(root), use_ripgrep=True, max_file_size_mb=1)])
search_modes_result = search_modes_agent.invoke({"messages": [("user", "Find handbook policy text.")]}, config=run_config())

In [ ]:
# Inspect output
show_messages(search_modes_result["messages"], pretty=True)
assert len([m for m in search_modes_result["messages"] if m.type == "tool"]) == 4
assert all(m.status == "success" for m in search_modes_result["messages"] if m.type == "tool")